# Day 1 homework — ingest and index your data (project)

This notebook follows **Day 1** in `course/`, but lives in **`project/`** and uses **different example repos** than the course notebook so you can practice on fresh documentation.

You will:

1. Download and process documentation from a GitHub repo (zip in memory)  
2. Parse **frontmatter** and markdown / MDX  
3. Run the same pipeline on **a repo you choose** for homework  

Run Jupyter from **this** folder:

```bash
uv run jupyter notebook
```

Dependencies match `course/`: `requests`, `python-frontmatter`, `jupyter` (dev).

## GitHub repo data

We download the repo as a **zip**, process text (`.md`, `.mdx`), and collect structured records for later indexing — same mental model as Day 1: prep ingredients before building the agent.

## Environment setup

Use **Python 3.10+** and **uv**. Install uv if needed: `pip install uv`.

Layout:

- `course/` — course examples (`Day_01_Ingest_and_Index_Your_Data.ipynb`)  
- `project/` — **this** homework project (`Day_01_homework.ipynb`)  

From `project/` you should already have:

```bash
uv init
uv add requests python-frontmatter
uv add --dev jupyter
```

## Understanding frontmatter

YAML between `---` lines, then Markdown body — common in Jekyll, Hugo, Next.js docs.

```markdown
---
title: "Getting Started with AI"
author: "John Doe"
date: "2024-01-15"
tags: ["ai", "machine-learning", "tutorial"]
difficulty: "beginner"
---

# Getting Started with AI

Body in **Markdown**.
```

In [1]:
import frontmatter

sample_md = """---
title: "Getting Started with AI"
author: "John Doe"
date: "2024-01-15"
tags: ["ai", "machine-learning", "tutorial"]
difficulty: "beginner"
---

# Getting Started with AI

This is the main content of the document written in **Markdown**.
"""

post = frontmatter.loads(sample_md)
print(post.metadata["title"])
print(post.metadata["tags"])
print(post.content[:80], "...")
post.to_dict()

Getting Started with AI
['ai', 'machine-learning', 'tutorial']
# Getting Started with AI

This is the main content of the document written in * ...


{'title': 'Getting Started with AI',
 'author': 'John Doe',
 'date': '2024-01-15',
 'tags': ['ai', 'machine-learning', 'tutorial'],
 'difficulty': 'beginner',
 'content': '# Getting Started with AI\n\nThis is the main content of the document written in **Markdown**.'}

## Sample repositories (homework — different from `course/`)

Use **other** repos than the course notebook (FAQ / Evidently). These use the **`main`** branch (same zip URL pattern as Day 1):

- [psf/requests](https://github.com/psf/requests) — classic HTTP library; Markdown under `docs/` and elsewhere  
- [pydantic/pydantic](https://github.com/pydantic/pydantic) — validation; many `.md` / `.mdx` docs in-repo  
- [astral-sh/uv](https://github.com/astral-sh/uv) — Python packaging tool; lots of docs as `.md` / `.mdx`  

**Note:** Some popular repos default to **`master`** (e.g. `encode/httpx`). The zip URL must use their real default branch, or you will get a 404. This notebook sticks to `main` so the code matches the course.

Download as zip via `requests`, open with `zipfile` + `io.BytesIO`, iterate `.md` / `.mdx`.

## Working with zip archives

URL pattern:

`https://codeload.github.com/{owner}/{repo}/zip/refs/heads/main`

In [2]:
import io
import zipfile

import frontmatter
import requests

# Homework walkthrough: psf/requests on main (different repos than course)
url = "https://codeload.github.com/psf/requests/zip/refs/heads/main"
resp = requests.get(url)
resp.raise_for_status()

repository_data = []
zf = zipfile.ZipFile(io.BytesIO(resp.content))

for file_info in zf.infolist():
    filename = file_info.filename.lower()
    if not filename.endswith(".md"):
        continue
    with zf.open(file_info) as f_in:
        content = f_in.read().decode("utf-8", errors="ignore")
        post = frontmatter.loads(content)
        data = post.to_dict()
        data["filename"] = filename
        repository_data.append(data)

zf.close()
len(repository_data), repository_data[1] if len(repository_data) > 1 else repository_data[0]

(12,
 {'content': '# Contribution Guidelines\n\nBefore opening any issues or proposing any pull requests, please read\nour [Contributor\'s Guide](https://requests.readthedocs.io/en/latest/dev/contributing/).\n\nTo get the greatest chance of helpful responses, please also observe the\nfollowing additional notes.\n\n## Questions\n\nThe GitHub issue tracker is for *bug reports* and *feature requests*. Please do\nnot use it to ask questions about how to use Requests. These questions should\ninstead be directed to [Stack Overflow](https://stackoverflow.com/). Make sure\nthat your question is tagged with the `python-requests` tag when asking it on\nStack Overflow, to ensure that it is answered promptly and accurately.\n\n## Good Bug Reports\n\nPlease be aware of the following things when filing bug reports:\n\n1. Avoid raising duplicate issues. *Please* use the GitHub issue search feature\n   to check whether your bug report or feature request has been mentioned in\n   the past. Duplicate bu

Many doc sites use **`.mdx`** (MDX). Include it in the filter when you need React-style docs, e.g. FastAPI’s docs tree:

```python
if not (filename.endswith(".md") or filename.endswith(".mdx")):
    continue
```

## Complete implementation — `read_repo_data`

In [3]:
import io
import zipfile

import frontmatter
import requests


def read_repo_data(repo_owner, repo_name):
    """
    Download and parse all markdown files from a GitHub repository.

    Args:
        repo_owner: GitHub username or organization
        repo_name: Repository name

    Returns:
        List of dictionaries containing file content and metadata
    """
    prefix = "https://codeload.github.com"
    url = f"{prefix}/{repo_owner}/{repo_name}/zip/refs/heads/main"
    resp = requests.get(url)

    if resp.status_code != 200:
        raise RuntimeError(f"Failed to download repository: {resp.status_code}")

    repository_data = []
    zf = zipfile.ZipFile(io.BytesIO(resp.content))

    for file_info in zf.infolist():
        filename = file_info.filename
        filename_lower = filename.lower()

        if not (filename_lower.endswith(".md") or filename_lower.endswith(".mdx")):
            continue

        try:
            with zf.open(file_info) as f_in:
                content = f_in.read().decode("utf-8", errors="ignore")
                post = frontmatter.loads(content)
                data = post.to_dict()
                data["filename"] = filename
                repository_data.append(data)
        except Exception as e:
            print(f"Error processing {filename}: {e}")
            continue

    zf.close()
    return repository_data

### Sanity check (homework repos — not the course examples)

In [4]:
pydantic_docs = read_repo_data("pydantic", "pydantic")
uv_docs = read_repo_data("astral-sh", "uv")

print(f"Pydantic documents: {len(pydantic_docs)}")
print(f"uv documents: {len(uv_docs)}")

Pydantic documents: 98
uv documents: 179


## Your homework — pick a repo

Replace **`YOUR_OWNER`** and **`YOUR_REPO`** with a documentation repo you care about (prefer many `.md` files). If the default branch is not `main`, you will need to adjust the download URL or use the GitHub API — Day 1 assumes `main`.

Optional: peek at one record with `my_data[0]`.

In [5]:
# Pick any docs repo you like (if default branch is not `main`, change the URL in read_repo_data)
YOUR_OWNER = "microsoft"  # change me
YOUR_REPO = "markitdown"  # change me

my_data = read_repo_data(YOUR_OWNER, YOUR_REPO)
print(f"Documents: {len(my_data)}")
# my_data[0]

Documents: 15


## Data processing considerations

- Small pages (e.g. short how-tos): often index as-is.  
- Large docs (big guides, generated API pages): plan for **chunking** in the next lesson (search quality, token limits).

## Homework checklist

1. Run this notebook end-to-end.  
2. Point `YOUR_OWNER` / `YOUR_REPO` at a real docs repo you chose.  
3. Post about what you built (learning in public).  

Cohort certificate: [Loom](https://www.loom.com/share/647233992ab44b0e94d756ab74dc1bb5)

## Learning in public

**LinkedIn (example)**  
Day 1 of AI Agents crash course — built a pipeline that downloads and parses GitHub docs. Repo: _yours_. Next: chunking. [Course](https://alexeygrigorev.com/aihero/)

**X (example)**  
Pipeline: zip → markdown/MDX → structured data for search. [Join](https://alexeygrigorev.com/aihero/)

## Community

[DataTalks.Club Slack](https://datatalks.club/) — `#course-ai-hero`